# NLA compressor model selection

Use this notebook after candidate validation runs finish. It discovers completed AE artifacts without duplicating evaluation logic and ranks only validation results. The final test split remains untouched until one configuration is frozen.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Run this notebook from within IAFlowCloud.')
    
    PROJECT_ROOT = PROJECT_ROOT.parent
code_path = PROJECT_ROOT / 'Code'
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))

from iaflow.comparison import collect_ae_results, smallest_qualified_model

results = collect_ae_results(PROJECT_ROOT)
print(f'Completed canonical AE runs: {len(results)}')
for result in results:
    print(
        f"{result['architecture']:>6s} {result['depth']:>7s} "
        f"latent={result['latent_dim']:02d} "
        f"variance={result['variance_recovered']:.8%} "
        f"log10 MSE={result['log10_mse']:.4e}"
    )

In [ ]:
selected = smallest_qualified_model(results, target_variance_recovered=0.999)
if selected is None:
    print('No canonical candidate reaches 99.9% validation variance recovery yet.')
else:
    print('Smallest qualified validation model:')
    print(selected)